# Random Forest and XGBoost on Morgan Fingerprints (Representation B)

This notebook uses Morgan fingerprints (refer to notebook 1b) and compares Random Forest and XGBoost models using k-fold cross-validation.

**Goal:** Compare Random Forest and XGBoost performance on fingerprint representation through k-fold cross-validation to determine which model performs best.



### Import Required Libraries

- **numpy**: Data manipulation and array operations
- **scipy.stats**: Statistical tests and confidence intervals
- **sklearn**: Machine learning tools (models, cross-validation, metrics)
- **xgboost**: Gradient boosting model (XGBClassifier)



In [ ]:
import numpy as np
from scipy import stats

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

# XGBoost
try:
    from xgboost import XGBClassifier
except ImportError:
    raise ImportError(
        "xgboost is not installed in this environment. Install it with `pip install xgboost` or `conda install -c conda-forge xgboost`."
    )

#ignore warnings
import warnings
warnings.filterwarnings('ignore')



In [ ]:
X_fp = np.load("training_fingerprints_matrix.npy")
y_fp = np.load("training_fingerprints_labels.npy")

print(f"Fingerprint matrix shape: {X_fp.shape}")
print(f"Labels shape: {y_fp.shape}")
print(f"Class balance: {(y_fp==1).sum()} active / {(y_fp==0).sum()} inactive")



Fingerprint matrix shape: (202895, 2048)
Labels shape: (202895,)
Class balance: 12514 active / 190381 inactive


## Step 2: Define Helper Function and Cross-Validation

Set random seed for reproducibility and define a helper function to calculate confidence intervals. This ensures that results are consistent across runs.



In [ ]:
seed = 20231124
np.random.seed(seed)

def calculate_ci(scores, confidence=0.95):
    n = len(scores)
    mean = np.mean(scores)
    std_err = stats.sem(scores)  # Standard error of the mean
    ci = stats.t.interval(confidence, n-1, loc=mean, scale=std_err)
    return mean, ci



## Step 3: Train Machine Learning Models

Train both **Random Forest** and **XGBoost** classifiers to predict molecular activity using fingerprints.

**Process:**

1. **5-Fold Cross-Validation**: 
   - Split data into 5 folds - use StratifiedKFold to distribute classes better
   - Train on 4 folds, test on 1 fold
   - Repeat 5 times with different splits
   - This gives a robust estimate of model performance

2. **XGBoost Classifier**
   - Gradient boosting model optimized for tabular data
   - Uses hyperparameters from previous tuning experiments

3. **Random Forest Classifier**:
   - Ensemble method that combines multiple decision trees
   - Good baseline model for structured data
   - Handles many features well

**Output:** Mean ROC AUC across all 5 folds with confidence intervals for both models




## 3a - XGBoost evaluation on Kfolds


In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=seed
)

xgb_clf = XGBClassifier(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.7,
    reg_alpha=0.3,
    reg_lambda=1.0,
    min_child_weight=1,
    gamma=0.0,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_auc = cross_val_score(
    xgb_clf,
    X_fp,
    y_fp,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1
)

xgb_mean, xgb_ci = calculate_ci(xgb_auc)

print("AUC per fold:", xgb_auc)
print("Mean AUC:", xgb_mean)
print("Std AUC:", xgb_auc.std())
print(f"95% CI: [{xgb_ci[0]:.4f}, {xgb_ci[1]:.4f}]")


## 3b - Random Forest evaluation on kfolds


In [ ]:
# Lightweight RF CV (mirrors the working setup), low parallelism
rf_acc = []
rf_auc = []

fold = 0
for train_index, test_index in cv.split(X_fp, y_fp):
    X_tr, X_te = X_fp[train_index], X_fp[test_index]
    y_tr, y_te = y_fp[train_index], y_fp[test_index]

    model_rf = RandomForestClassifier(
        random_state=42,
        n_jobs=1  # single-threaded to save RAM
    )
    model_rf.fit(X_tr, y_tr)

    preds = model_rf.predict(X_te)
    proba = model_rf.predict_proba(X_te)[:, 1]

    acc = accuracy_score(y_te, preds)
    auc = roc_auc_score(y_te, proba)

    rf_acc.append(acc)
    rf_auc.append(auc)

    fold += 1
    print(f"Done with fold {fold}/5")

rf_auc_array = np.array(rf_auc)  # convert to numpy array
rf_mean, rf_ci = calculate_ci(rf_auc_array)

print("\nRandom Forest on Fingerprints (lightweight)")
print("Accuracy per fold:", rf_acc)
print("Mean CV accuracy:", np.mean(rf_acc))
print("AUC per fold:", rf_auc)
print("Mean CV AUC:", rf_mean)
print(f"95% CI: [{rf_ci[0]:.4f}, {rf_ci[1]:.4f}]")


Done with fold 1/5
Done with fold 2/5
Done with fold 3/5
Done with fold 4/5
Done with fold 5/5

Random Forest on Fingerprints (lightweight)
Accuracy per fold: [0.9605707385593534, 0.960792528154957, 0.9603735922521501, 0.9619261194213756, 0.9615318268069691]
Mean CV accuracy: 0.961038961038961
AUC per fold: [0.9263362060305798, 0.9296841531521561, 0.9267523734623819, 0.9316208615634556, 0.9328345590292174]
Mean CV AUC: 0.9294456306475581
95% CI: [0.9259, 0.9330]


## 3c - Comparison and statistical test

Compare the performance of XGBoost and Random Forest on fingerprints using statistical tests to determine if the difference is significant.


In [ ]:
# display comparison
print("XGBoost results: ")
print("AUC per fold:", np.round(xgb_auc, 4))
print(f"Mean AUC: {xgb_mean:.4f}")
print(f"Std AUC: {xgb_auc.std():.4f}")
print(f"95% CI: [{xgb_ci[0]:.4f}, {xgb_ci[1]:.4f}]")
print(f"CI difference: {xgb_ci[1]-xgb_ci[0]:.4f}")
print("*******")
print("Random Forest results: ")
print("AUC per fold:", np.round(rf_auc_array, 4))
print(f"Mean AUC: {rf_mean:.4f}")
print(f"Std AUC: {rf_auc_array.std():.4f}")
print(f"95% CI: [{rf_ci[0]:.4f}, {rf_ci[1]:.4f}]")
print(f"CI difference: {rf_ci[1]-rf_ci[0]:.4f}")


XGBoost results: 
AUC per fold: [0.9174 0.9228 0.9211 0.9202 0.9216]
Mean AUC: 0.9206
Std AUC: 0.0018
95% CI: [0.9181, 0.9232]
CI difference: 0.0051
*******
Random Forest results: 
AUC per fold: [0.9263 0.9297 0.9268 0.9316 0.9328]
Mean AUC: 0.9294
Std AUC: 0.0026
95% CI: [0.9259, 0.9330]
CI difference: 0.0072


In [ ]:
# because of the overlap, use ttest to check if the results from the models performance mean it could be
# statistically significant
t_stat, p_value = stats.ttest_rel(xgb_auc, rf_auc_array)
print(f"Difference in means: {abs(xgb_mean - rf_mean):.4f}")
print(f"Paired t-test p-value: {p_value:.4f}")
if p_value < 0.05:
    winner = "XGBoost" if xgb_mean > rf_mean else "Random Forest"
    print(f"Winner: {winner} (statistically significant)")
else:
    print("No statistically significant difference between models")


Difference in means: 0.0088
Paired t-test p-value: 0.0015
Winner: Random Forest (statistically significant)


## 4 - Hold-out test evaluation
Train both models on a train split and evaluate on a held-out test set (20% stratified), plus a baseline dummy.


In [ ]:
# Train/test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_fp, y_fp,
    test_size=0.2,
    stratify=y_fp,
    random_state=seed,
)

# Train XGB with same params as CV
xgb_best = XGBClassifier(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.7,
    reg_alpha=0.3,
    reg_lambda=1.0,
    min_child_weight=1,
    gamma=0.0,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

xgb_best.fit(X_train, y_train)
xgb_proba_test = xgb_best.predict_proba(X_test)[:, 1]
xgb_test_auc = roc_auc_score(y_test, xgb_proba_test)

# Train RF lightweight
rf_best = RandomForestClassifier(
    random_state=42,
    n_jobs=1
)
rf_best.fit(X_train, y_train)
rf_proba_test = rf_best.predict_proba(X_test)[:, 1]
rf_test_auc = roc_auc_score(y_test, rf_proba_test)

# Baseline dummy
baseline = DummyClassifier(strategy='prior', random_state=seed)
baseline.fit(X_train, y_train)
baseline_proba = baseline.predict_proba(X_test)[:, 1]
baseline_test_auc = roc_auc_score(y_test, baseline_proba)

print("Test AUC (XGB):", round(xgb_test_auc, 4))
print("Test AUC (RF):", round(rf_test_auc, 4))
print("Baseline Test AUC:", round(baseline_test_auc, 4))


Test AUC (XGB): 0.9245
Test AUC (RF): 0.9294
Baseline Test AUC: 0.5


## 5 - Bootstrap CI on test AUC (best model)
Use bootstrap sampling on the held-out test predictions to get a 95% CI for AUC.


In [ ]:
def bootstrap_auc_ci(y_true, y_proba, n_samples=200, confidence=0.95, seed=seed):
    rng = np.random.default_rng(seed)
    aucs = []
    n = len(y_true)
    for _ in range(n_samples):
        idx = rng.choice(n, size=n, replace=True)
        aucs.append(roc_auc_score(y_true[idx], y_proba[idx]))
    aucs = np.array(aucs)
    mean = aucs.mean()
    std = aucs.std(ddof=1)
    alpha = 1 - confidence
    lower = np.percentile(aucs, alpha/2*100)
    upper = np.percentile(aucs, (1-alpha/2)*100)
    return mean, (lower, upper), std

# pick best model based on CV mean
best_name = "RF" if rf_mean >= xgb_mean else "XGB"
if best_name == "RF":
    best_proba = rf_proba_test
else:
    best_proba = xgb_proba_test

mean_boot, (ci_low, ci_up), std_boot = bootstrap_auc_ci(y_test.values if hasattr(y_test, 'values') else y_test, best_proba)

print(f"Best model by CV: {best_name}")
print(f"Test AUC (best): {mean_boot:.4f}")
print(f"Bootstrap 95% CI: [{ci_low:.4f}, {ci_up:.4f}]")
print(f"Bootstrap std: {std_boot:.4f}")


Best model by CV: RF
Test AUC (best): 0.9291
Bootstrap 95% CI: [0.9233, 0.9349]
Bootstrap std: 0.0033
